# Manufactura con CatBoost: clasificaci?n de piezas defectuosas

## Objetivo del ejercicio
En este ejercicio construiremos un modelo de **machine learning** para predecir si una pieza o lote ser? **defectuoso** (`1`) o **no defectuoso** (`0`) a partir de variables del proceso de manufactura.

## ?Qu? aprender? el alumno?
- C?mo cargar un dataset en Google Colab.
- C?mo explorar variables num?ricas y categ?ricas.
- C?mo entrenar un modelo con **CatBoost**.
- C?mo evaluar un modelo de clasificaci?n con m?tricas y visualizaciones.
- C?mo interpretar, de forma sencilla, los resultados del modelo.

## ?Por qu? usar CatBoost en este caso?
CatBoost es especialmente ?til cuando el dataset mezcla **variables num?ricas y categ?ricas**, porque:
- acepta variables categ?ricas de forma nativa;
- reduce el trabajo de codificaci?n manual;
- suele ofrecer buen desempe?o en datos tabulares;
- permite construir modelos competitivos con pocos ajustes iniciales.


## Ruta del ejercicio
1. Instalar e importar librer?as.
2. Cargar el dataset desde la computadora del alumno.
3. Realizar un an?lisis exploratorio breve.
4. Preparar los datos para el modelo.
5. Entrenar CatBoost.
6. Evaluar el desempe?o con m?tricas y gr?ficas.
7. Interpretar variables importantes.
8. Cerrar con conclusiones sobre las ventajas de CatBoost.


## 1. Instalaci?n e importaci?n de librer?as
En este bloque instalamos **CatBoost** y cargamos las librer?as que usaremos para an?lisis, visualizaci?n, partici?n de datos y evaluaci?n del modelo.


In [ ]:
%pip -q install catboost seaborn scikit-learn

import io
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import files
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
)

sns.set_theme(style='whitegrid', palette='Blues')
pd.set_option('display.max_columns', None)

print('Librer?as cargadas correctamente.')


## 2. Carga del dataset en Google Colab
Primero descargue el archivo `dataset_manufactura_catboost.csv` desde el portafolio.

Despu?s, ejecute este bloque para **subir el CSV manualmente** desde su computadora. Este flujo ayuda a que el alumno practique la carga de datos en Colab sin depender de rutas locales.


In [ ]:
print('Seleccione el archivo dataset_manufactura_catboost.csv desde su computadora.')
uploaded = files.upload()

if not uploaded:
    raise ValueError('No se carg? ning?n archivo. Vuelva a ejecutar el bloque y seleccione el CSV.')

uploaded_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[uploaded_name]))

print(f'Archivo cargado: {uploaded_name}')
print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
df.head()


## 3. Vista general de los datos
Aqu? revisamos el tama?o del dataset, los tipos de variables y si existen valores faltantes. Este paso permite entender la estructura de la informaci?n antes de modelar.


In [ ]:
print('Dimensiones del dataset:', df.shape)
print('
Tipos de datos:')
print(df.dtypes)

print('
Valores faltantes por columna:')
print(df.isna().sum())


## 4. Distribuci?n de la variable objetivo
La variable objetivo es `defectuoso`. Conviene revisar si las clases est?n balanceadas o si hay m?s observaciones de una categor?a que de otra.


In [ ]:
target_counts = df['defectuoso'].value_counts().sort_index()
target_pct = df['defectuoso'].value_counts(normalize=True).sort_index() * 100

summary_target = pd.DataFrame({
    'conteo': target_counts,
    'porcentaje': target_pct.round(2)
})
summary_target.index = ['No defectuoso (0)', 'Defectuoso (1)']
summary_target


In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='defectuoso', palette=['#9ecae1', '#3182bd'])
ax.set_title('Distribuci?n de la variable objetivo')
ax.set_xlabel('Defectuoso')
ax.set_ylabel('Frecuencia')
ax.set_xticklabels(['No', 'S?'])
plt.show()


## 5. Exploraci?n de variables num?ricas
En manufactura, variables como temperatura, presi?n, velocidad, vibraci?n y humedad pueden influir en la calidad. Primero las observamos con estad?sticos descriptivos y despu?s con histogramas.


In [ ]:
numeric_cols = ['temperatura', 'presion', 'velocidad_linea', 'vibracion', 'humedad']
df[numeric_cols].describe().T


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#3182bd')
    axes[i].set_title(f'Distribuci?n de {col}')

axes[-1].axis('off')
plt.tight_layout()
plt.show()


## 6. Comparaci?n num?rica seg?n la clase
Los diagramas de caja ayudan a comparar c?mo cambia cada variable num?rica entre piezas defectuosas y no defectuosas.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='defectuoso', y=col, ax=axes[i], palette=['#9ecae1', '#3182bd'])
    axes[i].set_title(f'{col} seg?n defectuoso')
    axes[i].set_xlabel('Defectuoso')
    axes[i].set_xticklabels(['No', 'S?'])

axes[-1].axis('off')
plt.tight_layout()
plt.show()


## 7. Exploraci?n de variables categ?ricas
Las variables `maquina`, `turno` y `material` son categ?ricas. CatBoost puede trabajarlas de forma nativa, lo cual es una de sus principales ventajas.


In [ ]:
categorical_cols = ['maquina', 'turno', 'material']

for col in categorical_cols:
    print(f'
Frecuencias de {col}:')
    print(df[col].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, categorical_cols):
    sns.countplot(data=df, x=col, hue='defectuoso', ax=ax, palette=['#9ecae1', '#3182bd'])
    ax.set_title(f'{col} por clase')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend(title='Defectuoso', labels=['No', 'S?'])

plt.tight_layout()
plt.show()


## 8. Correlaci?n entre variables num?ricas
Este mapa de calor solo se calcula con variables num?ricas. Sirve para identificar relaciones lineales aproximadas entre variables del proceso.


In [ ]:
plt.figure(figsize=(8, 5))
correlation_matrix = df[numeric_cols + ['defectuoso']].corr(numeric_only=True)
sns.heatmap(correlation_matrix, annot=True, cmap='Blues', fmt='.2f')
plt.title('Mapa de correlaci?n de variables num?ricas')
plt.show()


## 9. Preparaci?n de los datos
Ahora separamos variables predictoras (`X`) y variable objetivo (`y`). Tambi?n identificamos las columnas categ?ricas para indic?rselas a CatBoost.

Una ventaja importante es que **no necesitamos hacer one-hot encoding manual** en este ejemplo.


In [ ]:
X = df.drop(columns='defectuoso').copy()
y = df['defectuoso'].copy()

cat_features = [X.columns.get_loc(col) for col in categorical_cols]

print('Columnas predictoras:', X.columns.tolist())
print('Columnas categ?ricas:', categorical_cols)
print('?ndices de columnas categ?ricas para CatBoost:', cat_features)


## 10. Divisi?n en entrenamiento y prueba
Separamos los datos en entrenamiento y prueba para evaluar el modelo con observaciones que no participaron en el ajuste.

Usamos `stratify=y` para conservar la proporci?n de clases en ambos conjuntos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print('Tama?o de entrenamiento:', X_train.shape)
print('Tama?o de prueba:', X_test.shape)
print('Distribuci?n de la clase en entrenamiento:')
print(y_train.value_counts(normalize=True).round(3))
print('Distribuci?n de la clase en prueba:')
print(y_test.value_counts(normalize=True).round(3))


## 11. Entrenamiento del modelo CatBoost
Entrenaremos un `CatBoostClassifier` con hiperpar?metros simples y estables para principiantes.

- `iterations`: n?mero de ?rboles.
- `depth`: profundidad del ?rbol.
- `learning_rate`: velocidad de aprendizaje.
- `eval_metric`: m?trica principal de seguimiento.
- `verbose=False`: evita una salida extensa durante el entrenamiento.


In [ ]:
train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=250,
    depth=6,
    learning_rate=0.08,
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    verbose=False,
)

model.fit(train_pool)
print('Modelo entrenado correctamente.')


## 12. Predicciones y probabilidades
Generamos dos tipos de salida:
- la clase predicha (`0` o `1`);
- la probabilidad estimada de ser defectuoso.


In [ ]:
y_pred = model.predict(X_test).astype(int)
y_proba = model.predict_proba(X_test)[:, 1]

pred_preview = X_test.copy()
pred_preview['real'] = y_test.values
pred_preview['prediccion'] = y_pred
pred_preview['probabilidad_defectuoso'] = np.round(y_proba, 4)
pred_preview.head(10)


## 13. M?tricas de evaluaci?n
En clasificaci?n no basta con una sola m?trica. Aqu? calculamos:
- **accuracy**: proporci?n de aciertos totales;
- **precision**: calidad de las predicciones positivas;
- **recall**: capacidad para detectar defectuosos reales;
- **F1-score**: equilibrio entre precision y recall;
- **ROC AUC**: capacidad general para separar ambas clases.


In [ ]:
metrics_summary = pd.DataFrame({
    'M?trica': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC AUC'],
    'Valor': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_proba),
    ]
})

metrics_summary['Valor'] = metrics_summary['Valor'].round(4)
metrics_summary


In [ ]:
print('Reporte de clasificaci?n:')
print(classification_report(y_test, y_pred, target_names=['No defectuoso', 'Defectuoso']))


## 14. Matriz de confusi?n
La matriz de confusi?n muestra en qu? casos el modelo acierta y en cu?les se equivoca.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred. No', 'Pred. S?'],
            yticklabels=['Real No', 'Real S?'])
plt.title('Matriz de confusi?n')
plt.xlabel('Predicci?n')
plt.ylabel('Valor real')
plt.show()


## 15. Curva ROC
La curva ROC compara la tasa de verdaderos positivos contra la tasa de falsos positivos a distintos umbrales.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_value = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'CatBoost (AUC = {auc_value:.3f})', color='#2171b5')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Clasificador aleatorio')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curva ROC')
plt.legend()
plt.show()


## 16. Importancia de variables
Una forma sencilla de interpretar el modelo es revisar qu? variables tuvieron mayor peso en las predicciones.


In [ ]:
feature_importance = pd.DataFrame({
    'variable': X.columns,
    'importancia': model.get_feature_importance()
}).sort_values('importancia', ascending=False)

feature_importance


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=feature_importance, x='importancia', y='variable', palette='Blues_r')
plt.title('Importancia de variables en CatBoost')
plt.xlabel('Importancia')
plt.ylabel('Variable')
plt.show()


## 17. Interpretaci?n breve de resultados
Este bloque resume hallazgos iniciales del modelo y del dataset. El texto se genera a partir de las m?tricas y la importancia de variables para facilitar la interpretaci?n.


In [ ]:
top_features = feature_importance.head(3)['variable'].tolist()

print('Interpretaci?n general:')
print(f'- El dataset tiene {df.shape[0]} registros y {df.shape[1]} columnas.')
print('- La clase positiva (defectuoso = 1) representa aproximadamente el 16.7% del total, por lo que existe un desbalance moderado.')
print(f'- El ROC AUC del modelo fue de {auc_value:.3f}, lo que ayuda a evaluar la capacidad de separaci?n entre clases.')
print(f'- Las variables con mayor importancia fueron: {", ".join(top_features)}.')
print('- Esto no prueba causalidad, pero s? sugiere qu? variables fueron m?s ?tiles para distinguir piezas defectuosas de no defectuosas.')


## 18. Mini ejercicio para el alumno
Como pr?ctica adicional, cambie uno o dos hiperpar?metros del modelo, por ejemplo:
- `iterations`
- `depth`
- `learning_rate`

Despu?s compare si mejoran o empeoran las m?tricas.


In [ ]:
# Ejercicio opcional para el alumno
# Modifique los hiperpar?metros y vuelva a entrenar el modelo.
# Puede duplicar el bloque de entrenamiento y comparar nuevas m?tricas aqu?.


## 19. Conclusiones
### ?Qu? ventaja mostr? CatBoost en este ejercicio?
- Permiti? trabajar con variables categ?ricas como `maquina`, `turno` y `material` sin codificaci?n manual compleja.
- Ayud? a construir un flujo m?s simple para principiantes en problemas tabulares mixtos.
- Entreg? m?tricas ?tiles con pocos ajustes iniciales.
- Facilit? la interpretaci?n b?sica mediante importancia de variables.

### Cierre
CatBoost es una excelente alternativa cuando se desea construir modelos s?lidos para datos tabulares con mezcla de variables num?ricas y categ?ricas. En contextos industriales y de manufactura puede ser una herramienta muy ?til para apoyar decisiones de calidad, monitoreo y prevenci?n de defectos.
